# run_e25
CPU worker for Stage 4 E25 paired-error and label-stratified association analysis. Run cells 2, 3 and 4 in order. Configure only cell 3. Set `MODEL` (`'qwen'` or `'gemma'`), supply the explicit prepared Stage 4 plan and that model's four canonical E01 midpoint SCREEN prediction block references, and optionally supply explicit `RESUME_REFS` / `RESUME_FILE` or a `PARTS` subset (e.g. `['en']` for single-unit profiling). Completed analysis units stream their immutable references as they finish.


In [ ]:
import os, sys, subprocess
from pathlib import Path
if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use Python 3.11/3.12 with the pinned CPU requirements.')
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'
from google.colab import drive
drive.mount('/content/drive')
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
REPO = Path('/content/Research-stage4')
CODE_REF = 'phase1'  # Resolved once below; set this to the approved Stage 4 commit to pin explicitly.
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/Drazr/Research.git',str(REPO)],check=True)
if subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip():
    raise RuntimeError('Use a clean checkout; preserve local work before continuing.')
subprocess.run(['git','fetch','origin',CODE_REF],cwd=REPO,check=True)
CODE_COMMIT = subprocess.check_output(['git','rev-parse','FETCH_HEAD'],cwd=REPO,text=True).strip()
subprocess.run(['git','checkout','--detach',CODE_COMMIT],cwd=REPO,check=True)
subprocess.run([sys.executable,'-m','pip','install','-r',str(REPO/'requirements-stage2-cpu.txt')],check=True)
print('Fixed code commit:',CODE_COMMIT)
print('Restart if installation requests it, then rerun setup before continuing.')


In [ ]:
import json
MODEL = 'qwen'  # 'qwen' or 'gemma'
STAGE4_PLAN = PHASE1_ROOT / 'REPLACE_STAGE4_PLAN/_COMPLETE.json'
# Supply either PREDICTIONS_FILE (JSON list or {'predictions': [...]}) or explicit PREDICTION_REFS.
PREDICTIONS_FILE = None  # e.g. PHASE1_ROOT / 'REPLACE_E01_MIDPOINT_PREDICTIONS.json'
PREDICTION_REFS = []     # Exact {'path': ..., 'sha256': ...} dicts or _COMPLETE.json paths for the 4 midpoint SCREEN blocks.
# Optional explicit resume inputs from earlier completed E25 units for this MODEL.
RESUME_FILE = None
RESUME_REFS = []
# Optional subset of E25 parts: any of 'en', 'hi', 'bn', 'te', 'diagonal'. Empty runs all 5 units.
PARTS = []
# Optional path to write the completed E25 unit summary JSON.
OUTPUT_MANIFEST = None


In [ ]:
if MODEL not in ('qwen', 'gemma'):
    raise RuntimeError(f'Invalid MODEL selector: {MODEL!r}')
if subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() != CODE_COMMIT:
    raise RuntimeError('Checkout changed after setup.')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean Stage 4 checkout.')
if not STAGE4_PLAN.is_file():
    raise RuntimeError(f'Set the explicit mounted Stage 4 plan in cell 3: {STAGE4_PLAN}')
if PREDICTIONS_FILE is None and not PREDICTION_REFS:
    raise RuntimeError('Set PREDICTIONS_FILE or PREDICTION_REFS in cell 3 for the 4 midpoint SCREEN blocks.')
if PREDICTIONS_FILE is not None and not Path(PREDICTIONS_FILE).is_file():
    raise RuntimeError(f'Missing PREDICTIONS_FILE: {PREDICTIONS_FILE}')
if RESUME_FILE is not None and not Path(RESUME_FILE).is_file():
    raise RuntimeError(f'Missing RESUME_FILE: {RESUME_FILE}')
command = [
    sys.executable, '-u', '-m', 'src.stage4.run_e25',
    '--phase1-root', str(PHASE1_ROOT),
    '--manifest', str(STAGE4_PLAN),
    '--model', MODEL,
    '--storage-scope', 'persistent-drive',
]
if PREDICTIONS_FILE is not None:
    command += ['--predictions-file', str(PREDICTIONS_FILE)]
for ref in PREDICTION_REFS:
    command += ['--prediction-ref', json.dumps(ref) if isinstance(ref, dict) else str(ref)]
if RESUME_FILE is not None:
    command += ['--resume-file', str(RESUME_FILE)]
for ref in RESUME_REFS:
    command += ['--resume-ref', json.dumps(ref) if isinstance(ref, dict) else str(ref)]
for part in PARTS:
    command += ['--part', str(part)]
if OUTPUT_MANIFEST is not None:
    command += ['--output-manifest', str(OUTPUT_MANIFEST)]
process = subprocess.Popen(command, cwd=REPO, env=os.environ.copy(),
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in process.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    import signal
    process.send_signal(signal.SIGINT)
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    raise
if process.wait() != 0:
    raise RuntimeError('E25 worker blocked/failed. Preserve streamed references and inspect the failure before resuming.')
